Importo 
request: para las llamadas a la API
pandas: para transformar y analizar los datos
dotenv y os: Para leer la API key desde el archivo .env, sin exponerla en el código

In [16]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

Cargo el archivo .env para leer la API key

In [17]:
load_dotenv()
NASA_API_KEY = os.getenv("NASA_API_KEY")

Verifico que la conexion con la API esté funcionando correctamente

In [18]:
url = "https://api.nasa.gov/neo/rest/v1/feed"

params = {
    "start_date": "2024-01-01",
    "end_date": "2024-01-07",
    "api_key": NASA_API_KEY
}

response = requests.get(url, params=params)

print(response.status_code)

200


In [19]:
data = response.json()
print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['links', 'element_count', 'near_earth_objects'])


In [6]:
neos = data["near_earth_objects"]
print(type(neos))
print(neos.keys())

<class 'dict'>
dict_keys(['2024-01-02', '2024-01-01', '2024-01-04', '2024-01-03', '2024-01-06', '2024-01-05', '2024-01-07'])


In [7]:
primer_dia = neos["2024-01-01"]
print(type(primer_dia))
print(len(primer_dia))
print(primer_dia[0])

<class 'list'>
3
{'links': {'self': 'http://api.nasa.gov/neo/rest/v1/neo/3724393?api_key=QTiPpbhjU5Acq8of527QVIM7XQ8teOgd1hUEZcO5'}, 'id': '3724393', 'neo_reference_id': '3724393', 'name': '(2015 OD22)', 'nasa_jpl_url': 'https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=3724393', 'absolute_magnitude_h': 21.21, 'estimated_diameter': {'kilometers': {'estimated_diameter_min': 0.152249185, 'estimated_diameter_max': 0.3404395273}, 'meters': {'estimated_diameter_min': 152.249185036, 'estimated_diameter_max': 340.4395272595}, 'miles': {'estimated_diameter_min': 0.0946032284, 'estimated_diameter_max': 0.2115392495}, 'feet': {'estimated_diameter_min': 499.5052162336, 'estimated_diameter_max': 1116.9276186141}}, 'is_potentially_hazardous_asteroid': False, 'close_approach_data': [{'close_approach_date': '2024-01-01', 'close_approach_date_full': '2024-Jan-01 11:58', 'epoch_date_close_approach': 1704110280000, 'relative_velocity': {'kilometers_per_second': '25.3931645932', 'kilometers_per_hour

In [8]:
asteroids = []

for date, neo_list in neos.items():
    for neo in neo_list:
        asteroid = {
            "id": neo["id"],
            "name": neo["name"],
            "is_potentially_hazardous": neo["is_potentially_hazardous_asteroid"],
            "diameter_km_min": neo["estimated_diameter"]["kilometers"]["estimated_diameter_min"],
            "diameter_km_max": neo["estimated_diameter"]["kilometers"]["estimated_diameter_max"],
            "close_approach_date": neo["close_approach_data"][0]["close_approach_date"],
            "miss_distance_km": neo["close_approach_data"][0]["miss_distance"]["kilometers"],
            "velocity_kmh": neo["close_approach_data"][0]["relative_velocity"]["kilometers_per_hour"]
        }
        asteroids.append(asteroid)

print(len(asteroids))

45


In [9]:
df = pd.DataFrame(asteroids)
df.head()

,id,name,is_potentially_hazardous,diameter_km_min,diameter_km_max,close_approach_date,miss_distance_km,velocity_kmh
0,2415949,415949 (2001 XY10),False,0.355267,0.794401,2024-01-02,50452409.349026638,57205.8951204341
1,3160747,(2003 SR84),False,0.016771,0.037501,2024-01-02,19798169.933318188,38589.054833182
2,3309828,(2005 YQ96),True,0.199781,0.446725,2024-01-02,24984732.559194501,56413.0143519451
3,3457842,(2009 HC21),False,0.101054,0.225964,2024-01-02,73609796.924990823,21891.1182185894
4,3553062,(2010 XA11),False,0.016016,0.035813,2024-01-02,35275514.131770482,31468.9783588978


In [10]:
print(df.shape)
print(df["is_potentially_hazardous"].value_counts())

(45, 8)
is_potentially_hazardous
False    39
True      6
Name: count, dtype: int64


In [11]:
print(df.dtypes)
print(df.isnull().sum())

id                              str
name                            str
is_potentially_hazardous       bool
diameter_km_min             float64
diameter_km_max             float64
close_approach_date             str
miss_distance_km                str
velocity_kmh                    str
dtype: object
id                          0
name                        0
is_potentially_hazardous    0
diameter_km_min             0
diameter_km_max             0
close_approach_date         0
miss_distance_km            0
velocity_kmh                0
dtype: int64


In [12]:
df["miss_distance_km"] = pd.to_numeric(df["miss_distance_km"])
df["velocity_kmh"] = pd.to_numeric(df["velocity_kmh"])
df["close_approach_date"] = pd.to_datetime(df["close_approach_date"])

print(df.dtypes)

id                                     str
name                                   str
is_potentially_hazardous              bool
diameter_km_min                    float64
diameter_km_max                    float64
close_approach_date         datetime64[us]
miss_distance_km                   float64
velocity_kmh                       float64
dtype: object


In [13]:
import time
from datetime import datetime, timedelta

def fetch_neos(start_date, end_date, api_key):
    url = "https://api.nasa.gov/neo/rest/v1/feed"
    params = {
        "start_date": start_date,
        "end_date": end_date,
        "api_key": api_key
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json()["near_earth_objects"]
    else:
        print(f"Error {response.status_code} para {start_date}")
        return {}

In [14]:
start = datetime(1995, 1, 1)
end = datetime(2025, 1, 1)

all_asteroids = []
current = start

while current < end:
    week_end = current + timedelta(days=7)
    print(f"Bajando: {current.date()} a {week_end.date()}")
    current = week_end

Bajando: 1995-01-01 a 1995-01-08
Bajando: 1995-01-08 a 1995-01-15
Bajando: 1995-01-15 a 1995-01-22
Bajando: 1995-01-22 a 1995-01-29
Bajando: 1995-01-29 a 1995-02-05
Bajando: 1995-02-05 a 1995-02-12
Bajando: 1995-02-12 a 1995-02-19
Bajando: 1995-02-19 a 1995-02-26
Bajando: 1995-02-26 a 1995-03-05
Bajando: 1995-03-05 a 1995-03-12
Bajando: 1995-03-12 a 1995-03-19
Bajando: 1995-03-19 a 1995-03-26
Bajando: 1995-03-26 a 1995-04-02
Bajando: 1995-04-02 a 1995-04-09
Bajando: 1995-04-09 a 1995-04-16
Bajando: 1995-04-16 a 1995-04-23
Bajando: 1995-04-23 a 1995-04-30
Bajando: 1995-04-30 a 1995-05-07
Bajando: 1995-05-07 a 1995-05-14
Bajando: 1995-05-14 a 1995-05-21
Bajando: 1995-05-21 a 1995-05-28
Bajando: 1995-05-28 a 1995-06-04
Bajando: 1995-06-04 a 1995-06-11
Bajando: 1995-06-11 a 1995-06-18
Bajando: 1995-06-18 a 1995-06-25
Bajando: 1995-06-25 a 1995-07-02
Bajando: 1995-07-02 a 1995-07-09
Bajando: 1995-07-09 a 1995-07-16
Bajando: 1995-07-16 a 1995-07-23
Bajando: 1995-07-23 a 1995-07-30
Bajando: 1

In [15]:
start = datetime(1995, 1, 1)
end = datetime(2025, 1, 1)

all_asteroids_raw = {}
current = start

while current < end:
    week_end = current + timedelta(days=7)
    start_str = current.strftime("%Y-%m-%d")
    end_str = week_end.strftime("%Y-%m-%d")
    
    neos_week = fetch_neos(start_str, end_str, NASA_API_KEY)
    all_asteroids_raw.update(neos_week)
    
    current = week_end
    time.sleep(0.5)

print(f"Total de días bajados: {len(all_asteroids_raw)}")

KeyboardInterrupt: 

In [ ]:
import json

with open("../data/raw/neo_raw.json", "w") as f:
    json.dump(all_asteroids_raw, f)

print("Datos guardados correctamente")

In [ ]:
import os
size_mb = os.path.getsize("../data/raw/neo_raw.json") / (1024 * 1024)
print(f"Tamaño del archivo: {size_mb:.2f} MB")

In [ ]:
all_asteroids = []

for date, neo_list in all_asteroids_raw.items():
    for neo in neo_list:
        asteroid = {
            "id": neo["id"],
            "name": neo["name"],
            "is_potentially_hazardous": neo["is_potentially_hazardous_asteroid"],
            "diameter_km_min": neo["estimated_diameter"]["kilometers"]["estimated_diameter_min"],
            "diameter_km_max": neo["estimated_diameter"]["kilometers"]["estimated_diameter_max"],
            "close_approach_date": neo["close_approach_data"][0]["close_approach_date"],
            "miss_distance_km": neo["close_approach_data"][0]["miss_distance"]["kilometers"],
            "velocity_kmh": neo["close_approach_data"][0]["relative_velocity"]["kilometers_per_hour"]
        }
        all_asteroids.append(asteroid)

df = pd.DataFrame(all_asteroids)
print(df.shape)

In [ ]:
df["miss_distance_km"] = pd.to_numeric(df["miss_distance_km"])
df["velocity_kmh"] = pd.to_numeric(df["velocity_kmh"])
df["close_approach_date"] = pd.to_datetime(df["close_approach_date"])

print(df.dtypes)
print(df.isnull().sum())

In [ ]:
df.to_csv("../data/processed/neo_clean.csv", index=False)
print("Guardado correctamente")

In [20]:
df = pd.read_csv("../data/processed/neo_clean.csv")
print(df.shape)
df.head()

(79439, 8)


,id,name,is_potentially_hazardous,diameter_km_min,diameter_km_max,close_approach_date,miss_distance_km,velocity_kmh
0,3516633,(2010 HA),False,0.044112,0.098637,1995-01-07,2.959363e+07,22395.337325
1,3837644,(2019 AY3),False,0.045767,0.102339,1995-01-07,1.909375e+07,80679.829203
2,3843493,(2019 PY),False,0.023150,0.051765,1995-01-07,1.440650e+07,17995.842471
3,2446862,446862 (2001 VB76),True,0.208236,0.465629,1995-01-08,7.622845e+06,27326.492203
4,3765015,(2016 WR48),False,0.160160,0.358129,1995-01-08,4.339162e+07,26721.139515
